# 🏎️ JetBot Mario Kart — Combined Vision + Racing

**Track:** Oval — clockwise  
**Vision:** Jake's Canny + HoughLinesP tape detector  
**Control:** Oval racing controller (PID + clockwise bias, phase-aware speed)

### Architecture
```
Camera → ROI crop → Canny → HoughLinesP → filter by angle → LaneInput
                                                                ↓
                         OvalRacingController (PID + phase) ←───┘
                                    ↓
                              JetBot motors
```

The main loop runs in a **background thread** so the buttons (MOTOR TEST, START, STOP)
stay responsive while vision and motor commands run continuously.

### Run order
1. Run Cells 1–6 to define classes
2. Run Cell 7 — opens camera, shows preview, starts background loop, shows controls
3. Click **MOTOR TEST** to verify motors physically work
4. Tune sliders until the green arrow tracks the lane
5. Click **START** to race, **STOP** to halt
6. Run Cell 8 when done to clean up

## Cell 1 — Imports

In [1]:
import time
import threading
from dataclasses import dataclass
from enum import Enum, auto
from typing import Optional

import numpy as np
import cv2

from jetbot import Robot

import ipywidgets as widgets
from IPython.display import display

print("✅ Imports OK")

✅ Imports OK


## Cell 2 — Shared data structures

In [2]:
@dataclass
class LaneInput:
    offset: float = 0.0
    lane_detected: bool = True


@dataclass
class MotorCommand:
    left: float = 0.0
    right: float = 0.0


class TrackPhase(Enum):
    STRAIGHT = auto()
    CORNER   = auto()


class RaceState(Enum):
    WAITING   = auto()
    RACING    = auto()
    SEARCHING = auto()
    STOPPED   = auto()


print("✅ Data structures defined")

✅ Data structures defined


## Cell 3 — PID + Oval Racing Controller

In [3]:
class PIDController:
    """PID with asymmetric integral clamp for clockwise oval."""

    def __init__(self, Kp, Ki, Kd, integral_max_right=1.0, integral_max_left=0.3):
        self.Kp = Kp; self.Ki = Ki; self.Kd = Kd
        self.integral_max_right = integral_max_right
        self.integral_max_left  = integral_max_left
        self._integral = 0.0
        self._prev_error = 0.0
        self._prev_time = None

    def reset(self):
        self._integral = 0.0
        self._prev_error = 0.0
        self._prev_time = None

    def compute(self, error):
        now = time.monotonic()
        if self._prev_time is None:
            dt = 0.033
        else:
            dt = max(now - self._prev_time, 1e-6)

        self._integral += error * dt
        self._integral = max(-self.integral_max_left,
                             min(self.integral_max_right, self._integral))

        derivative = (error - self._prev_error) / dt
        self._prev_error = error
        self._prev_time = now

        return self.Kp * error + self.Ki * self._integral + self.Kd * derivative


class OvalRacingController:
    """Clockwise oval racing controller — no confidence threshold."""

    def __init__(self, straight_speed=0.45, corner_speed=0.30, max_speed=0.60,
                 steering_gain=0.60, clockwise_bias=0.08, corner_threshold=0.25,
                 speed_slew_rate=0.05, pid=None):
        self.straight_speed   = straight_speed
        self.corner_speed     = corner_speed
        self.max_speed        = max_speed
        self.steering_gain    = steering_gain
        self.clockwise_bias   = clockwise_bias
        self.corner_threshold = corner_threshold
        self.speed_slew_rate  = speed_slew_rate
        self.pid = pid or PIDController(Kp=0.6, Ki=0.02, Kd=0.1)
        self._current_speed = straight_speed
        self.current_phase  = TrackPhase.STRAIGHT

    def compute(self, lane):
        if not lane.lane_detected:
            return MotorCommand(0.0, 0.0)

        if abs(lane.offset) > self.corner_threshold:
            self.current_phase = TrackPhase.CORNER
            target_speed = self.corner_speed
        else:
            self.current_phase = TrackPhase.STRAIGHT
            target_speed = self.straight_speed

        delta = target_speed - self._current_speed
        delta = max(-self.speed_slew_rate, min(self.speed_slew_rate, delta))
        self._current_speed += delta
        speed = self._current_speed

        steering = self.pid.compute(lane.offset) + self.clockwise_bias

        left_speed  = speed + steering * self.steering_gain
        right_speed = speed - steering * self.steering_gain

        left_speed  = max(-self.max_speed, min(self.max_speed, left_speed))
        right_speed = max(-self.max_speed, min(self.max_speed, right_speed))

        return MotorCommand(left=left_speed, right=right_speed)

    def reset(self):
        self.pid.reset()
        self._current_speed = self.straight_speed
        self.current_phase  = TrackPhase.STRAIGHT


print("✅ Controllers defined")

✅ Controllers defined


## Cell 4 — Vision → LaneInput adapter (Canny + HoughLinesP)

In [4]:
class LaneDetector:
    """Canny + HoughLinesP detector with side-line fitting, EMA smoothing, and hold-on-miss."""

    def __init__(self, canny_lo=50, canny_hi=150, blur_k=5,
                 min_line_len=40, max_line_gap=10,
                 min_angle=20, roi_pct=40,
                 ema_alpha=0.35, hold_frames=15):
        self.canny_lo     = canny_lo
        self.canny_hi     = canny_hi
        self.blur_k       = blur_k
        self.min_line_len = min_line_len
        self.max_line_gap = max_line_gap
        self.min_angle    = min_angle
        self.roi_pct      = roi_pct

        self.ema_alpha = ema_alpha
        self.hold_frames = hold_frames

        self.ema = {'l': None, 'r': None}
        self.last_data_y = {'l': None, 'r': None}
        self.stale = {'l': 0, 'r': 0}

    def _fit_side_linear(self, side_segs):
        """Deg-1 fit x = m*y + b with 2-sigma outlier rejection."""
        if len(side_segs) < 1:
            return None

        xs, ys = [], []
        for _, (x1, y1, x2, y2) in side_segs:
            xs.extend([x1, x2])
            ys.extend([y1, y2])

        xs_arr = np.asarray(xs, dtype=np.float64)
        ys_arr = np.asarray(ys, dtype=np.float64)

        if np.ptp(ys_arr) < 2 or len(xs_arr) < 2:
            return None

        m, b = np.polyfit(ys_arr, xs_arr, 1)
        resid = xs_arr - (m * ys_arr + b)
        sigma = float(np.std(resid))

        if sigma > 1.0 and len(xs_arr) >= 4:
            mask = np.abs(resid) < 2.0 * sigma
            if mask.sum() >= 2 and np.ptp(ys_arr[mask]) > 1:
                m, b = np.polyfit(ys_arr[mask], xs_arr[mask], 1)
                xs_arr, ys_arr = xs_arr[mask], ys_arr[mask]

        return float(m), float(b), float(ys_arr.min()), float(ys_arr.max())

    def _polyline(self, vis, xs, ys, color, thickness):
        pts = np.stack([xs, ys], axis=1).astype(np.int32)
        cv2.polylines(vis, [pts], isClosed=False, color=color, thickness=thickness)

    def detect(self, frame):
        h, w = frame.shape[:2]
        center_x = w // 2
        bot_pt = (center_x, h - 1)

        k = self.blur_k if self.blur_k % 2 == 1 else self.blur_k + 1

        roi_h = max(1, int(h * self.roi_pct / 100))
        roi_y0 = h - roi_h

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (k, k), 0)
        roi_gray = gray[roi_y0:, :]

        edges = cv2.Canny(roi_gray, self.canny_lo, self.canny_hi)

        lines = cv2.HoughLinesP(
            edges, rho=1, theta=np.pi / 180, threshold=40,
            minLineLength=self.min_line_len, maxLineGap=self.max_line_gap,
        )

        vis = frame.copy()
        cv2.line(vis, (center_x, 0), (center_x, h - 1), (0, 255, 255), 1)
        cv2.line(vis, (0, roi_y0), (w, roi_y0), (255, 255, 255), 1)

        kept = []
        if lines is not None:
            for seg in lines[:, 0, :]:
                x1, y1, x2, y2 = seg
                dx = x2 - x1
                dy = y2 - y1
                angle = abs(np.degrees(np.arctan2(dy, dx)))
                if angle > 90:
                    angle = 180 - angle
                if angle < self.min_angle:
                    continue

                mx = (x1 + x2) // 2
                y1f = y1 + roi_y0
                y2f = y2 + roi_y0
                kept.append((mx, (x1, y1f, x2, y2f)))

        buckets = {
            'l': [k for k in kept if k[0] < center_x],
            'r': [k for k in kept if k[0] >= center_x],
        }

        fits = {}
        for side in ('l', 'r'):
            raw = self._fit_side_linear(buckets[side])
            if raw is None:
                self.stale[side] += 1
                if self.ema[side] is not None and self.stale[side] <= self.hold_frames and self.last_data_y[side] is not None:
                    fits[side] = (self.ema[side][0], self.ema[side][1], self.last_data_y[side])
                else:
                    fits[side] = None
                continue

            m, b, y_min, y_max = raw
            if self.ema[side] is None:
                self.ema[side] = (m, b)
            else:
                em, eb = self.ema[side]
                a = self.ema_alpha
                self.ema[side] = ((1 - a) * em + a * m, (1 - a) * eb + a * b)

            self.last_data_y[side] = (y_min, y_max)
            self.stale[side] = 0
            fits[side] = (self.ema[side][0], self.ema[side][1], self.last_data_y[side])

        lf, rf = fits.get('l'), fits.get('r')

        lane_detected = lf is not None and rf is not None
        offset = 0.0

        if lane_detected:
            lm, lb, (ly_min, ly_max) = lf
            rm, rb, (ry_min, ry_max) = rf

            ly = np.linspace(ly_min, ly_max, 20)
            lx = lm * ly + lb
            ry = np.linspace(ry_min, ry_max, 20)
            rx = rm * ry + rb

            self._polyline(vis, lx, ly, (255, 0, 0), 2)
            self._polyline(vis, rx, ry, (0, 128, 255), 2)

            y_top = max(ly_min, ry_min)
            y_bot = max(min(ly_max, ry_max), h - 1)

            if y_bot > y_top:
                mid_y = np.linspace(y_top, y_bot, 20)
                mid_x = ((lm * mid_y + lb) + (rm * mid_y + rb)) / 2.0
                self._polyline(vis, mid_x, mid_y, (0, 255, 0), 3)

                bot_mx = int(((lm * (h - 1) + lb) + (rm * (h - 1) + rb)) / 2.0)
                err_px = bot_mx - center_x
                offset = float(np.clip(err_px / (w / 2.0), -1.0, 1.0))

                look_mx = int(mid_x[0])
                look_my = int(mid_y[0])
                cv2.arrowedLine(vis, bot_pt, (look_mx, look_my), (0, 255, 0), 3, tipLength=0.08)

                held = []
                if self.stale['l'] > 0:
                    held.append(f"Lhold{self.stale['l']}")
                if self.stale['r'] > 0:
                    held.append(f"Rhold{self.stale['r']}")
                tag = (' ' + ' '.join(held)) if held else ''

                cv2.putText(vis, f"off={offset:+.2f}{tag}",
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            else:
                cv2.putText(vis, "NO Y OVERLAP",
                            (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                lane_detected = False
                offset = 0.0
        else:
            missing = []
            if lf is None:
                missing.append('NO LEFT')
            if rf is None:
                missing.append('NO RIGHT')
            cv2.putText(vis, ' '.join(missing),
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        edges_full = np.zeros((h, w), dtype=np.uint8)
        edges_full[roi_y0:, :] = edges
        edges_bgr = cv2.cvtColor(edges_full, cv2.COLOR_GRAY2BGR)
        cv2.line(edges_bgr, (0, roi_y0), (w, roi_y0), (255, 255, 255), 1)

        lane = LaneInput(offset=offset, lane_detected=lane_detected)
        return lane, vis, edges_bgr


print("✅ LaneDetector defined")

✅ LaneDetector defined


## Cell 5 — Tunable parameters

In [5]:
# ── Racing parameters ─────────────────────────────────────────
STRAIGHT_SPEED         = 0.2
CORNER_SPEED           = 0.28
MAX_SPEED              = 0.4
CLOCKWISE_BIAS         = 0.08
CORNER_THRESHOLD       = 0.25
KP                     = 0.6
KI                     = 0.02
KD                     = 0.1
STEERING_GAIN          = 0.6
SPEED_SLEW_RATE        = 0.05

# Recovery — non-blocking (duration tracked by wall clock in the loop)
RECOVERY_SPIN_SPEED    = 0.25
RECOVERY_SPIN_DURATION = 0.4
LANE_LOST_GIVEUP_SEC   = 3.0     # stop if lane is lost for this long

LAP_TIME_ESTIMATE      = 12.0

# ── Vision defaults ───────────────────────────────────────────
VISION_CANNY_LO     = 50
VISION_CANNY_HI     = 150
VISION_BLUR_K       = 5
VISION_MIN_LINE_LEN = 40
VISION_MAX_LINE_GAP = 10
VISION_MIN_ANGLE    = 20
VISION_ROI_PCT      = 40

# ── Camera ────────────────────────────────────────────────────
CAM_WIDTH  = 640
CAM_HEIGHT = 480
CAM_FPS    = 30

print("Parameters set.")

Parameters set.


## Cell 6 — RaceSession (non-blocking)

The recovery spin is now **non-blocking**. Instead of `time.sleep()` freezing the loop,
`_recovery_spin()` just sets motor values and records when it started — the main loop
then cancels the spin after `RECOVERY_SPIN_DURATION` has elapsed. This keeps the
vision preview responsive throughout.

In [6]:
class RaceSession:
    """Manages race lifecycle — non-blocking so buttons stay responsive."""

    def __init__(self, robot):
        self.robot = robot
        self.controller = OvalRacingController(
            straight_speed=STRAIGHT_SPEED, corner_speed=CORNER_SPEED,
            max_speed=MAX_SPEED, steering_gain=STEERING_GAIN,
            clockwise_bias=CLOCKWISE_BIAS, corner_threshold=CORNER_THRESHOLD,
            speed_slew_rate=SPEED_SLEW_RATE,
            pid=PIDController(Kp=KP, Ki=KI, Kd=KD,
                              integral_max_right=1.0, integral_max_left=0.3),
        )
        self.state = RaceState.WAITING
        self.lap_count = 0
        self._race_start = None
        self._last_lap_t = None
        self._lost_since = None
        self._spin_started = None   # wall-clock time the recovery spin began
        self.last_reason = "Waiting — click START"

    # ── User-facing controls ─────────────────────────────────

    def start(self):
        self.controller.reset()
        self.lap_count = 0
        self._lost_since = None
        self._spin_started = None
        self._race_start = time.monotonic()
        self._last_lap_t = self._race_start
        self.state = RaceState.RACING
        self.last_reason = "Racing"
        print("🏎️  Race started — clockwise oval.")

    def stop(self):
        self.state = RaceState.STOPPED
        self.robot.stop()
        self._spin_started = None
        self.last_reason = "Stopped by user"
        elapsed = time.monotonic() - self._race_start if self._race_start else 0
        print(f"🛑 Stopped after {elapsed:.1f} s, {self.lap_count} laps.")

    def motor_test(self):
        """Blocking — only runs when called from a button click, not the loop."""
        print("🔧 Motor test — forward at 0.30 for 0.5 s...")
        self.robot.left_motor.value  = 0.30
        self.robot.right_motor.value = 0.30
        time.sleep(0.5)
        self.robot.stop()
        print("   Done. If wheels didn't turn, check battery / motor drivers.")

    # ── Per-frame step ───────────────────────────────────────

    def step(self, lane):
        """Called every vision frame. Returns the MotorCommand being applied."""
        now = time.monotonic()

        if self.state == RaceState.STOPPED:
            self.last_reason = "State = STOPPED"
            return MotorCommand(0, 0)

        if self.state == RaceState.WAITING:
            self.last_reason = "Click START to race"
            return MotorCommand(0, 0)

        # Lap counting by timer
        if (self.state == RaceState.RACING
            and LAP_TIME_ESTIMATE > 0
            and self._last_lap_t is not None
            and (now - self._last_lap_t) >= LAP_TIME_ESTIMATE):
            self._record_lap()

        if self.state == RaceState.RACING:
            if not lane.lane_detected:
                # Enter SEARCHING, start non-blocking recovery spin
                self._lost_since = now
                self._spin_started = now
                self.state = RaceState.SEARCHING
                self.last_reason = "Lane lost — searching"
                self.robot.left_motor.value  =  RECOVERY_SPIN_SPEED
                self.robot.right_motor.value = -RECOVERY_SPIN_SPEED
                return MotorCommand(RECOVERY_SPIN_SPEED, -RECOVERY_SPIN_SPEED)

            cmd = self.controller.compute(lane)
            self.robot.left_motor.value  = cmd.left
            self.robot.right_motor.value = cmd.right
            self.last_reason = "Racing"
            return cmd

        elif self.state == RaceState.SEARCHING:
            # Re-acquired?
            if lane.lane_detected:
                self.controller.reset()
                self.state = RaceState.RACING
                self._spin_started = None
                self.last_reason = "Racing"
                print("🔍 Lane re-acquired.")
                cmd = self.controller.compute(lane)
                self.robot.left_motor.value  = cmd.left
                self.robot.right_motor.value = cmd.right
                return cmd

            # Still lost — manage the spin
            if self._spin_started is not None:
                if (now - self._spin_started) < RECOVERY_SPIN_DURATION:
                    # Keep spinning
                    return MotorCommand(RECOVERY_SPIN_SPEED, -RECOVERY_SPIN_SPEED)
                else:
                    # Spin complete — idle until next frame decides
                    self.robot.stop()
                    self._spin_started = None
                    return MotorCommand(0, 0)

            # Give up after too long
            if (now - (self._lost_since or now)) > LANE_LOST_GIVEUP_SEC:
                self.stop()
                print(f"⚠️  Lane lost for {LANE_LOST_GIVEUP_SEC} s — stopping.")

        return MotorCommand(0, 0)

    def lap_complete(self):
        self._record_lap()

    def _record_lap(self):
        self.lap_count += 1
        self._last_lap_t = time.monotonic()
        print(f"✅ Lap {self.lap_count} complete!")


print("✅ RaceSession defined")

✅ RaceSession defined


## Cell 7 — Everything live: hardware, UI, background loop

Runs once. After this cell finishes:
- The camera preview is live
- Sliders update the detector in real time
- Buttons work because the loop is in a **background thread**
- The cell itself returns immediately so Jupyter's kernel stays responsive

If you need to restart, run Cell 8 (cleanup) first, then re-run this cell.

In [7]:
# ── GStreamer pipeline ────────────────────────────────────────
def gstreamer_pipeline(width=CAM_WIDTH, height=CAM_HEIGHT, fps=CAM_FPS):
    return (
        f"nvarguscamerasrc ! "
        f"video/x-raw(memory:NVMM), width={width}, height={height}, "
        f"format=NV12, framerate={fps}/1 ! "
        f"nvvidconv flip-method=0 ! "
        f"video/x-raw, width={width}, height={height}, format=BGRx ! "
        f"videoconvert ! video/x-raw, format=BGR ! appsink"
    )

# ── Hardware + core objects ───────────────────────────────────
cap = cv2.VideoCapture(gstreamer_pipeline(), cv2.CAP_GSTREAMER)
assert cap.isOpened(), "Camera failed to open!"

robot = Robot()
race_session = RaceSession(robot)
detector = LaneDetector(
    canny_lo=VISION_CANNY_LO, canny_hi=VISION_CANNY_HI,
    blur_k=VISION_BLUR_K,
    min_line_len=VISION_MIN_LINE_LEN, max_line_gap=VISION_MAX_LINE_GAP,
    min_angle=VISION_MIN_ANGLE, roi_pct=VISION_ROI_PCT,
)

# ── Widgets ───────────────────────────────────────────────────
image_widget = widgets.Image(format='jpeg', width=1280, height=480)

wide  = widgets.Layout(width='500px')
style = {'description_width': '120px'}
canny_lo_s     = widgets.IntSlider(value=VISION_CANNY_LO,     min=0,  max=255, description='canny lo',      layout=wide, style=style, continuous_update=True)
canny_hi_s     = widgets.IntSlider(value=VISION_CANNY_HI,     min=0,  max=500, description='canny hi',      layout=wide, style=style, continuous_update=True)
blur_s         = widgets.IntSlider(value=VISION_BLUR_K,       min=1,  max=21, step=2, description='blur k', layout=wide, style=style, continuous_update=True)
min_line_len_s = widgets.IntSlider(value=VISION_MIN_LINE_LEN, min=5,  max=300, description='min line len',  layout=wide, style=style, continuous_update=True)
max_line_gap_s = widgets.IntSlider(value=VISION_MAX_LINE_GAP, min=0,  max=100, description='max line gap',  layout=wide, style=style, continuous_update=True)
min_angle_s    = widgets.IntSlider(value=VISION_MIN_ANGLE,    min=0,  max=89,  description='min angle deg', layout=wide, style=style, continuous_update=True)
roi_pct_s      = widgets.IntSlider(value=VISION_ROI_PCT,      min=20, max=100, description='roi % (bottom)',layout=wide, style=style, continuous_update=True)

status_lbl = widgets.Label(value='Status: WAITING — click START to race')
laps_lbl   = widgets.Label(value='Laps: 0')
phase_lbl  = widgets.Label(value='Phase: —')
motor_lbl  = widgets.Label(value='L: 0.00   R: 0.00')
fps_lbl    = widgets.Label(value='FPS: —')

btn_test  = widgets.Button(description='MOTOR TEST', button_style='info',
                           layout=widgets.Layout(width='140px', height='40px'))
btn_start = widgets.Button(description='START', button_style='success',
                           layout=widgets.Layout(width='120px', height='40px'))
btn_stop  = widgets.Button(description='STOP', button_style='danger',
                           layout=widgets.Layout(width='120px', height='40px'))

def on_test(_):
    race_session.motor_test()

def on_start(_):
    race_session.start()
    status_lbl.value = 'Status: RACING'

def on_stop(_):
    race_session.stop()
    status_lbl.value = 'Status: STOPPED'

btn_test.on_click(on_test)
btn_start.on_click(on_start)
btn_stop.on_click(on_stop)

display(image_widget)
display(widgets.VBox([
    widgets.HBox([btn_test, btn_start, btn_stop, status_lbl]),
    widgets.HBox([laps_lbl, phase_lbl, motor_lbl, fps_lbl]),
    canny_lo_s, canny_hi_s, blur_s,
    min_line_len_s, max_line_gap_s, min_angle_s, roi_pct_s,
]))

# ── Background loop ───────────────────────────────────────────
stop_flag = threading.Event()

def bgr_to_jpeg(frame):
    _, buf = cv2.imencode('.jpeg', frame)
    return bytes(buf)

def loop():
    frame_count = 0
    t_last = time.monotonic()
    while not stop_flag.is_set():
        ret, frame = cap.read()
        if not ret:
            time.sleep(0.01)
            continue

        # Push slider values into the detector
        detector.canny_lo     = canny_lo_s.value
        detector.canny_hi     = canny_hi_s.value
        detector.blur_k       = blur_s.value
        detector.min_line_len = min_line_len_s.value
        detector.max_line_gap = max_line_gap_s.value
        detector.min_angle    = min_angle_s.value
        detector.roi_pct      = roi_pct_s.value

        # Vision + race step
        try:
            lane, vis, edges_bgr = detector.detect(frame)
            cmd = race_session.step(lane)
        except Exception as e:
            print(f"Loop error: {e}")
            race_session.stop()
            break

        # Dashboard
        status_lbl.value = f"Status: {race_session.state.name} ({race_session.last_reason})"
        laps_lbl.value   = f"Laps: {race_session.lap_count}"
        phase_lbl.value  = f"Phase: {race_session.controller.current_phase.name}"
        motor_lbl.value  = f"L: {cmd.left:+.2f}   R: {cmd.right:+.2f}"

        frame_count += 1
        if frame_count % 15 == 0:
            now = time.monotonic()
            fps = 15 / max(now - t_last, 1e-6)
            fps_lbl.value = f"FPS: {fps:.1f}"
            t_last = now

        # Preview
        combined = np.hstack([vis, edges_bgr])
        image_widget.value = bgr_to_jpeg(combined)

        time.sleep(0.01)  # yield briefly

    # Loop exit — ensure motors off
    robot.stop()
    print("Background loop exited.")

loop_thread = threading.Thread(target=loop, daemon=True)
loop_thread.start()

print("✅ Background loop running.")
print("   1. Click MOTOR TEST to verify motors")
print("   2. Tune sliders until the green arrow tracks the lane")
print("   3. Click START to race")

Image(value=b'', format='jpeg', height='480', width='1280')

✅ Background loop running.
   1. Click MOTOR TEST to verify motors
   2. Tune sliders until the green arrow tracks the lane
   3. Click START to race


## Cell 8 — Cleanup

Stops the background loop, cuts motors, releases the camera.  
Run this before re-running Cell 7 (e.g. after tweaking parameters in Cell 5).

In [ ]:
stop_flag.set()
loop_thread.join(timeout=2.0)
race_session.stop()
cap.release()
print("Camera released, background loop stopped, motors off.")